# Stage C2 — Composite Loss Assembly

Dual-fuel PINN pipeline | Sandrine Schueller Mafra | PPGEM – UFPR
Supports dissertation Sec. 3.4 (Eq. 3.16–3.20).

C1 built and unit-tested six penalty *formulas*. This notebook turns
them into one trainable objective: variance-weighted data loss
(Eq. 3.17), the six physics terms combined with C1's adaptive weight
$\lambda_j(x)$ (Eq. 3.18), weight-decay regularization (Eq. 3.19),
weight calibration so no term dominates at initialization, and the
sigmoid schedule (Eq. 3.20) that introduces physics gradually.

**Notation note, easy to conflate:** Eq. 3.16 has *two* different
weights per constraint — $\lambda_j(x)$ (C1's density/validity
modulation, varies **by collocation point**, fixed once computed) and
$w_j$ (Eq. 3.16's outer scalar, varies **by epoch** via the Eq. 3.20
schedule, same value for every point at a given epoch). $\lambda_j(x)$
is already folded into $\mathcal{L}_{\text{physics},j}$ itself
(Eq. 3.18); $w_j(t)$ multiplies that whole term from outside.

**Input:** `data/masters_data.xlsx`,
`outputs/C1_collocation_points.csv`, `outputs/B1_selected_architecture.json`
**Output:** a single `total_loss(model, t)` function C4 trains against,
plus the calibrated $w_j^{\text{final}}$ values and schedule parameters.


## Setup

In [ ]:
import json
import numpy as np
import polars as pl
import plotly.graph_objects as go
from pathlib import Path
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print("polars    ", pl.__version__)
import plotly
print("plotly    ", plotly.__version__)
print("tensorflow", tf.__version__)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "code" else Path.cwd()
RAW_PATH = PROJECT_ROOT / "data" / "masters_data.xlsx"
OUT_DIR = PROJECT_ROOT / "outputs"
SEED = 42
RAW_PATH


## 1. Load data, architecture, and C1's collocation points

Reuses C1's 1000 LHS points and their precomputed $\rho(x)/\rho_{\max}$
and $v_{\eta\text{-NOx}}(x)$ instead of regenerating them — same
collocation set throughout C1-C4 matters more than re-deriving it
matters little.

In [ ]:
COLUMN_MAP = {
    "SOI [o.CA]": "SOI", "Lambda [-]": "lambda", "Sub. Rate [%]": "sub_rate",
    "Prail [bar]": "P_rail", "HC [g/kW.h]": "HC", "NOX [ppm]": "NOx",
    "CO2 [%]": "CO2", "SO_H [FSN]": "PM", "ETA [%]": "eta",
}
INPUT_COLS = ["SOI", "lambda", "sub_rate", "P_rail"]
OUTPUT_COLS = ["HC", "NOx", "CO2", "PM", "eta"]
ALL_COLS = INPUT_COLS + OUTPUT_COLS
N_IN, N_OUT = len(INPUT_COLS), len(OUTPUT_COLS)
SOI_IDX, LAMBDA_IDX, SUBRATE_IDX, PRAIL_IDX = [INPUT_COLS.index(c) for c in INPUT_COLS]
HC_IDX, NOX_IDX, CO2_IDX, PM_IDX, ETA_IDX = [OUTPUT_COLS.index(c) for c in OUTPUT_COLS]
EMISSION_IDXS = [HC_IDX, NOX_IDX, CO2_IDX, PM_IDX]

df = pl.read_excel(RAW_PATH).rename(COLUMN_MAP).select(ALL_COLS)
n = df.shape[0]
medians = {c: df[c].median() for c in INPUT_COLS}
ranges = {c: (df[c].max() - df[c].min()) for c in INPUT_COLS}
deviation = np.column_stack([np.abs(df[c].to_numpy() - medians[c]) / ranges[c] for c in INPUT_COLS])
raw_block = np.array(INPUT_COLS)[deviation.argmax(axis=1)]

def smooth_isolated_labels(labels, passes=2):
    out = list(labels)
    for _ in range(passes):
        changed = False
        for i in range(1, len(out) - 1):
            if out[i] != out[i - 1] and out[i - 1] == out[i + 1]:
                out[i] = out[i - 1]
                changed = True
        if not changed:
            break
    return np.array(out)

ofat_block = smooth_isolated_labels(raw_block)
extremity = np.zeros(n)
for b in np.unique(ofat_block):
    idx = np.where(ofat_block == b)[0]
    vals = df[b].to_numpy()[idx]
    order = np.argsort(vals)
    m = len(idx)
    pos = np.array([0.5]) if m == 1 else np.empty(m)
    if m > 1:
        ranks = np.empty(m)
        ranks[order] = np.arange(m)
        pos = ranks / (m - 1)
    extremity[idx] = np.abs(pos - 0.5) * 2
rng_np = np.random.default_rng(SEED)
jitter = rng_np.uniform(-1e-9, 1e-9, size=n)
order = np.argsort(-(extremity + jitter))
split = np.array(["train"] * n)
split[order[:6]] = "test"
split[order[6:12]] = "val"
df = df.with_columns(pl.Series("split", split))
train_df = df.filter(pl.col("split") == "train")
train_min = {c: train_df[c].min() for c in ALL_COLS}
train_max = {c: train_df[c].max() for c in ALL_COLS}
df = df.with_columns([
    ((pl.col(c) - train_min[c]) / (train_max[c] - train_min[c])).alias(f"{c}_norm")
    for c in ALL_COLS
])
X_train = df.filter(pl.col("split") == "train").select([f"{c}_norm" for c in INPUT_COLS]).to_numpy().astype(np.float32)
Y_train = df.filter(pl.col("split") == "train").select([f"{c}_norm" for c in OUTPUT_COLS]).to_numpy().astype(np.float32)

colloc_path = OUT_DIR / "C1_collocation_points.csv"
if colloc_path.exists():
    colloc_df = pl.read_csv(colloc_path)
    X_colloc = colloc_df.select(INPUT_COLS).to_numpy().astype(np.float32)
    rho_colloc = colloc_df["density_ratio"].to_numpy()
    validity_eff_nox = colloc_df["validity_eff_nox"].to_numpy()
    print(f"Loaded {X_colloc.shape[0]} collocation points from C1")
else:
    raise FileNotFoundError("Run C1's save cell first -- outputs/C1_collocation_points.csv not found.")

selection_path = OUT_DIR / "B1_selected_architecture.json"
if selection_path.exists():
    with open(selection_path) as f:
        HIDDEN_UNITS = tuple(json.load(f)["hidden_units"])
else:
    HIDDEN_UNITS = ()
    print("WARNING: B1 selection not found, using linear fallback.")
print("architecture hidden units:", HIDDEN_UNITS)


## 2. Data loss (Eq. 3.17) — variance-weighted MSE

$$
\mathcal{L}_{\text{data}} = \frac{1}{N}\sum_{i=1}^{N}\sum_{k=1}^{5} \frac{1}{\sigma_k^2}\left(y^{\text{pred}}_{i,k} - y^{\text{exp}}_{i,k}\right)^2
$$

$\sigma_k^2$ from the **training data**, normalized outputs, held
fixed for the rest of the pipeline — not recomputed per batch.

In [ ]:
sigma2 = Y_train.var(axis=0, ddof=1)
sigma2 = np.maximum(sigma2, 1e-8)  # guard against a near-constant output
print("per-output variance (normalized):", dict(zip(OUTPUT_COLS, np.round(sigma2, 5))))

def data_loss(predict_fn, X, Y, sigma2=sigma2):
    X_t = tf.convert_to_tensor(X, dtype=tf.float32)
    Y_t = tf.convert_to_tensor(Y, dtype=tf.float32)
    pred = predict_fn(X_t)
    sq_err = tf.square(pred - Y_t) / tf.constant(sigma2, dtype=tf.float32)
    return tf.reduce_mean(tf.reduce_sum(sq_err, axis=1))


## 3. Physics loss (Eq. 3.18) — C1's formulas, now weighted by $\lambda_j(x)$

Same six formulas as C1, refactored to return the **per-point** penalty
(no averaging yet) so the pointwise weight can be applied first:

$$
\mathcal{L}_{\text{physics},j} = \frac{1}{N_c}\sum_{c=1}^{N_c} \lambda_j(x_c)\, p_j(x_c)
$$

Non-negativity (Eq. 3.14) is the one exception — its own formula has
no $\lambda_j(x)$ term, so it stays a plain mean, exactly as in C1.

In [ ]:
def monotonic_decreasing_per_point(predict_fn, x, out_idx, in_idx):
    x_t = tf.convert_to_tensor(x, dtype=tf.float32)
    with tf.GradientTape() as tape:
        tape.watch(x_t)
        target = predict_fn(x_t)[:, out_idx]
    d = tape.gradient(target, x_t)[:, in_idx]
    return tf.square(tf.maximum(0.0, d))

def convexity_per_point(predict_fn, x, out_idx, in_idx):
    x_t = tf.convert_to_tensor(x, dtype=tf.float32)
    with tf.GradientTape() as tape2:
        tape2.watch(x_t)
        with tf.GradientTape() as tape1:
            tape1.watch(x_t)
            target = predict_fn(x_t)[:, out_idx]
        d_first = tape1.gradient(target, x_t)[:, in_idx]
    d_second = tape2.gradient(d_first, x_t)[:, in_idx]
    return tf.square(tf.maximum(0.0, -d_second))

def tradeoff_per_point(predict_fn, x, out_idx_a, out_idx_b, in_idx):
    x_t = tf.convert_to_tensor(x, dtype=tf.float32)
    with tf.GradientTape(persistent=True) as tape:
        tape.watch(x_t)
        y = predict_fn(x_t)
        a, b = y[:, out_idx_a], y[:, out_idx_b]
    grad_a = tape.gradient(a, x_t)[:, in_idx]
    grad_b = tape.gradient(b, x_t)[:, in_idx]
    del tape
    return tf.square(tf.maximum(0.0, grad_a * grad_b))

def nonneg_loss(predict_fn, x, out_idxs=EMISSION_IDXS):
    x_t = tf.convert_to_tensor(x, dtype=tf.float32)
    y = predict_fn(x_t)
    emissions = tf.gather(y, out_idxs, axis=1)
    return tf.reduce_mean(tf.reduce_sum(tf.square(tf.maximum(0.0, -emissions)), axis=1))

def weighted_physics_loss(per_point, lambda_x):
    return tf.reduce_mean(tf.constant(lambda_x, dtype=tf.float32) * per_point)

CONSTRAINT_NAMES = ["NOx-SOI", "PM-lambda", "HC-lambda", "NOx-PM", "eta-NOx"]


## 4. Regularization (Eq. 3.19) — weight decay, biases excluded

$$
\mathcal{L}_{\text{reg}} = \frac{1}{P}\sum_{l=1}^{L}\sum_{i,j}\left(w^{(l)}_{ij}\right)^2
$$

$P$ is the model's **total** parameter count (Sec. 3.4.1.3 — same
number C1/B1 already compute via `count_params`), even though the sum
itself only touches kernels (2D weight matrices), not biases (1D).

In [ ]:
def regularization_loss(model):
    P = model.count_params()
    sq_sum = tf.add_n([tf.reduce_sum(tf.square(w)) for w in model.trainable_weights if len(w.shape) > 1])
    return sq_sum / P


## 5. Weight calibration — set $w_j^{\text{final}}$ so physics matches data at full strength

Build a fresh (untrained) network, evaluate $\mathcal{L}_{\text{data}}$
and every $\mathcal{L}_{\text{physics},j}$ on it once, then set

$$
w_j^{\text{final}} = \frac{\mathcal{L}_{\text{data}}}{\mathcal{L}_{\text{physics},j}}
$$

so that **once the schedule reaches full strength**, $w_j^{\text{final}}\cdot\mathcal{L}_{\text{physics},j} \approx \mathcal{L}_{\text{data}}$
— this is the target the schedule ramps *toward*, not the weight
used at epoch 0 (Section 6 handles that distinction).

In [ ]:
tf.random.set_seed(SEED)
calib_model = keras.Sequential([keras.Input(shape=(N_IN,))])
for units in HIDDEN_UNITS:
    calib_model.add(layers.Dense(units, activation="tanh"))
calib_model.add(layers.Dense(N_OUT, activation="linear"))
predict_fn = lambda x: calib_model(x, training=False)

L_data_init = float(data_loss(predict_fn, X_train, Y_train))

per_point_fns = {
    "NOx-SOI": lambda: monotonic_decreasing_per_point(predict_fn, X_colloc, NOX_IDX, SOI_IDX),
    "PM-lambda": lambda: monotonic_decreasing_per_point(predict_fn, X_colloc, PM_IDX, LAMBDA_IDX),
    "HC-lambda": lambda: convexity_per_point(predict_fn, X_colloc, HC_IDX, LAMBDA_IDX),
    "NOx-PM": lambda: tradeoff_per_point(predict_fn, X_colloc, NOX_IDX, PM_IDX, SOI_IDX),
    "eta-NOx": lambda: tradeoff_per_point(predict_fn, X_colloc, ETA_IDX, NOX_IDX, SOI_IDX),
}
validity_by_constraint = {
    "NOx-SOI": np.ones(len(X_colloc)), "PM-lambda": np.ones(len(X_colloc)),
    "HC-lambda": np.ones(len(X_colloc)), "NOx-PM": np.ones(len(X_colloc)),
    "eta-NOx": validity_eff_nox,
}

L_physics_init, w_final = {}, {}
for name, fn in per_point_fns.items():
    lambda_x = rho_colloc * validity_by_constraint[name]  # lambda_j^0 folded in at =1 here; C3 tunes it
    L_physics_init[name] = float(weighted_physics_loss(fn(), lambda_x))
    w_final[name] = L_data_init / max(L_physics_init[name], 1e-12)

L_nonneg_init = float(nonneg_loss(predict_fn, X_colloc))
w_final["non-negativity"] = L_data_init / max(L_nonneg_init, 1e-12)
L_physics_init["non-negativity"] = L_nonneg_init

L_reg_init = float(regularization_loss(calib_model))

print(f"L_data (init) = {L_data_init:.5f}\n")
for name in list(per_point_fns) + ["non-negativity"]:
    print(f"  {name:14s} L_physics(init)={L_physics_init[name]:.6f}   "
          f"w_final={w_final[name]:10.2f}   check: w_final*L_physics = {w_final[name]*L_physics_init[name]:.5f}")
print(f"\nL_reg (init) = {L_reg_init:.6f}")


## 6. Sigmoid weight schedule (Eq. 3.20)

$$
w_j(t) = \frac{w_j^{\text{final}}}{1 + \exp[-k(t-t_0)]}
$$

Illustrative defaults ($k$, $t_0$) below — Table 8 (C3) tunes these as
hyperparameters; the point here is just confirming the mechanics: near
0 early, $w_j^{\text{final}}/2$ at $t_0$, approaching $w_j^{\text{final}}$
late.

In [ ]:
K_SCHEDULE = 0.01
T0_SCHEDULE = 750

def scheduled_weight(t, w_j_final, k=K_SCHEDULE, t0=T0_SCHEDULE):
    return w_j_final / (1 + np.exp(-k * (t - t0)))

for name in list(w_final):
    w0 = scheduled_weight(0, w_final[name])
    w_mid = scheduled_weight(T0_SCHEDULE, w_final[name])
    print(f"{name:14s} w(0)={w0:9.4f}   w(t0)={w_mid:9.4f} (should be w_final/2={w_final[name]/2:9.4f})   "
          f"w_final={w_final[name]:9.2f}")


**The schedule itself**, all six constraints:

In [ ]:
t_range = np.linspace(0, 2000, 300)
fig = go.Figure()
palette = ["#185FA5", "#993C1D", "#534AB7", "#3B6D11", "#854F0B", "#5A5A55"]
for (name, wf), color in zip(w_final.items(), palette):
    fig.add_trace(go.Scatter(x=t_range, y=scheduled_weight(t_range, wf), mode="lines",
                              name=name, line=dict(color=color)))
fig.add_vline(x=T0_SCHEDULE, line_dash="dot", line_color="#B0AFA8", annotation_text="t0")
fig.update_layout(title="w_j(t): sigmoid ramp per constraint", xaxis_title="epoch",
                   yaxis_title="w_j(t)", yaxis_type="log", width=800, height=450)
fig.show()


## 7. Composite loss (Eq. 3.16) — the function C4 actually trains against

$$
\mathcal{L}_{\text{total}}(t) = w_{\text{data}}\mathcal{L}_{\text{data}} + \sum_{j=1}^{M} w_j(t)\,\mathcal{L}_{\text{physics},j} + w_{\text{reg}}\mathcal{L}_{\text{reg}}
$$

$w_{\text{data}}=1$ and $w_{\text{reg}}=10^{-5}$, both fixed per Sec. 3.4.1.

In [ ]:
W_DATA = 1.0
W_REG = 1e-5

def total_loss(model, X, Y, X_colloc, rho_colloc, validity_eff_nox, epoch, w_final):
    predict_fn = lambda x: model(x, training=True)
    l_data = data_loss(predict_fn, X, Y)

    l_phys = 0.0
    for name, fn in [
        ("NOx-SOI", lambda: monotonic_decreasing_per_point(predict_fn, X_colloc, NOX_IDX, SOI_IDX)),
        ("PM-lambda", lambda: monotonic_decreasing_per_point(predict_fn, X_colloc, PM_IDX, LAMBDA_IDX)),
        ("HC-lambda", lambda: convexity_per_point(predict_fn, X_colloc, HC_IDX, LAMBDA_IDX)),
        ("NOx-PM", lambda: tradeoff_per_point(predict_fn, X_colloc, NOX_IDX, PM_IDX, SOI_IDX)),
        ("eta-NOx", lambda: tradeoff_per_point(predict_fn, X_colloc, ETA_IDX, NOX_IDX, SOI_IDX)),
    ]:
        validity = validity_eff_nox if name == "eta-NOx" else np.ones(len(X_colloc))
        lambda_x = rho_colloc * validity
        l_j = weighted_physics_loss(fn(), lambda_x)
        l_phys += float(scheduled_weight(epoch, w_final[name])) * l_j

    l_nonneg = nonneg_loss(predict_fn, X_colloc)
    l_phys += float(scheduled_weight(epoch, w_final["non-negativity"])) * l_nonneg

    l_reg = regularization_loss(model)
    return W_DATA * l_data + l_phys + W_REG * l_reg

# smoke test on the calibration model, epoch 0 and epoch 2000
for ep in [0, 2000]:
    lt = float(total_loss(calib_model, X_train, Y_train, X_colloc, rho_colloc, validity_eff_nox, ep, w_final))
    print(f"total_loss at epoch {ep}: {lt:.5f}")


## 8. Does physics vanish or dominate? (feedback: does the physics loss disappear or take over)

**Caveat up front:** this fixes $\mathcal{L}_{\text{data}}$ and each
raw $\mathcal{L}_{\text{physics},j}$ at their *initialization* values
and only varies $w_j(t)$ — it shows the schedule's intended shape, not
a real training trajectory (both loss values will themselves change
once the network actually trains, which only **C4** can show). What it
does confirm here: the mechanism doesn't let physics swamp the data
term early on, and does reach comparable magnitude late.

In [ ]:
t_range2 = np.linspace(0, 2000, 100)
data_contribution = np.full_like(t_range2, W_DATA * L_data_init)
physics_contribution = np.zeros_like(t_range2)
for name in w_final:
    physics_contribution += scheduled_weight(t_range2, w_final[name]) * L_physics_init[name]
reg_contribution = np.full_like(t_range2, W_REG * L_reg_init)

fig = go.Figure()
fig.add_trace(go.Scatter(x=t_range2, y=data_contribution, mode="lines", name="data term (fixed L_data_init)",
                          line=dict(color="#185FA5", width=2)))
fig.add_trace(go.Scatter(x=t_range2, y=physics_contribution, mode="lines",
                          name="physics terms (schedule x fixed L_physics_init)",
                          line=dict(color="#993C1D", width=2)))
fig.add_trace(go.Scatter(x=t_range2, y=reg_contribution, mode="lines", name="reg term",
                          line=dict(color="#3B6D11", width=1, dash="dot")))
fig.update_layout(title="Illustrative loss-component magnitudes under the schedule (not a real training run)",
                   xaxis_title="epoch", yaxis_title="loss contribution", yaxis_type="log",
                   width=800, height=450)
fig.show()


## Optional — persist outputs

In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
config = {
    "w_data": W_DATA, "w_reg": W_REG,
    "w_final": w_final, "k_schedule": K_SCHEDULE, "t0_schedule": T0_SCHEDULE,
    "L_data_init": L_data_init, "L_physics_init": L_physics_init, "L_reg_init": L_reg_init,
}
with open(OUT_DIR / "C2_loss_config.json", "w") as f:
    json.dump(config, f, indent=2)
print(f"Saved to {OUT_DIR}")


## Next

**C3** searches Table 8's hyperparameter space, including the
$\lambda_j^0$ scaling and, per the text, $k$/$t_0$ for this schedule —
the illustrative values here (`K_SCHEDULE`, `T0_SCHEDULE`) are a
starting point, not the tuned result. **C4** trains the ensemble
against `total_loss` exactly as assembled here.
